In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
img=cv2.imread("C:/Users/SHAGUN PRAJAPATI/Screenshot 2026-08-05 095414.png")
print(type(img))
print(img.shape)
cv2.imshow("photos",img)
cv2.waitKey(0)
img_resize=cv2.resize(img,(400,256))
cv2.imshow("photos",img_resize)
cv2.waitKey(0)
img_flip=cv2.flip(img,0) #horizontal
cv2.imshow("photos",img_flip)
cv2.waitKey(0)
img_flip=cv2.flip(img,1) #vertical
cv2.imshow("photos",img_flip)
cv2.waitKey(0)
img_crop=img[100:300,200:500]
cv2.imshow("photos",img_crop)
cv2.waitKey(0)
cv2.destroyAllWindows()

<class 'numpy.ndarray'>
(1080, 1920, 3)


In [3]:
import cv2
from ultralytics import settings

# Disable data and crash report tracking
settings.update({"sync": False})
from ultralytics import YOLO

# 1. Load the lightweight YOLOv8 Nano model (~6MB, runs fast on CPU)
model = YOLO('yolov8n.pt')

# 2. Open the webcam (0 is default built-in camera)
cap = cv2.VideoCapture(0)
print("Starting Object Counter... Press 'q' on the video window to quit.")

while cap.isOpened():
  success, frame = cap.read()
  if not success:
    print("Failed to grab camera frame.")
    break

  # 3. Perform object detection on the current frame
  results = model(frame, verbose=False)

  # Get total number of objects detected in this frame
  detected_count = len(results[0].boxes)

  # 4. Render detection boxes directly on the frame
  annotated_frame = results[0].plot()

  # 5. Display the object count overlay on screen
  cv2.putText(
      annotated_frame,
      f'Total Objects Counted: {detected_count}',
      (20, 50),  # Position (x, y)
      cv2.FONT_HERSHEY_SIMPLEX,
      1,  # Font scale
      (0, 255, 0),  # Text color (Green)
      2,  # Thickness
      cv2.LINE_AA,
  )

  # 6. Show the frame in a window
  cv2.imshow('BCA AI Demo - Live Object Counter', annotated_frame)

  # 7. Terminate program cleanly when pressing 'q'
  if cv2.waitKey(1) & 0xFF == ord('q'):
    print("Terminating program...")
    break

# Release camera and close all windows cleanly
cap.release()

# 2. Add short wait key loops to let Windows pump destroy events cleanly
cv2.waitKey(1)
cv2.destroyAllWindows()
cv2.waitKey(1)

print("Camera feed closed successfully.")


Starting Object Counter... Press 'q' on the video window to quit.
Terminating program...
Camera feed closed successfully.


In [ ]:
import cv2
import numpy as np

# 1. Initialize Webcam (CAP_DSHOW prevents camera light freeze on Windows)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# 2. Define HSV Color Range for a YELLOW object (Default)
# Tip: Adjust lower/upper bounds if using a Red/Green object!
LOWER_COLOR = np.array([20, 100, 100])  # Lower bound for Yellow in HSV
UPPER_COLOR = np.array([35, 255, 255])  # Upper bound for Yellow in HSV

canvas = None
prev_point = None

print("Air Canvas Started! Wave a YELLOW marker in front of the camera to draw.")
print("Press 'c' to clear canvas. Press 'q' to quit.")

while cap.isOpened():
  success, frame = cap.read()
  if not success:
    break

  # Flip frame horizontally for intuitive "mirror" drawing
  frame = cv2.flip(frame, 1)

  # Initialize black canvas to match camera size on first frame
  if canvas is None:
    canvas = np.zeros_like(frame)

  # --- CORE CV CONCEPTS FOR STUDENTS ---

  # A. Convert BGR to HSV color space (easier to isolate specific colors)
  hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

  # B. Create binary mask (White = color detected, Black = background)
  mask = cv2.inRange(hsv, LOWER_COLOR, UPPER_COLOR)

  # Clean noise using morphological erosion and dilation
  mask = cv2.erode(mask, None, iterations=1)
  mask = cv2.dilate(mask, None, iterations=1)

  # C. Find contours (outlines of detected shapes)
  contours, _ = cv2.findContours(
      mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
  )

  curr_point = None

  if contours:
    # Get the largest contour (prevents small background noise from drawing)
    largest_contour = max(contours, key=cv2.contourArea)

    # Filter out tiny detections by minimum area threshold
    if cv2.contourArea(largest_contour) > 500:
      # Calculate centroid (center x, y coordinates) of detected marker
      M = cv2.moments(largest_contour)
      if M['m00'] != 0:
        cX = int(M['m10'] / M['m00'])
        cY = int(M['m01'] / M['m00'])
        curr_point = (cX, cY)

        # Draw a visual tracking dot around object tip
        cv2.circle(frame, curr_point, 8, (0, 255, 255), -1)

  # D. Draw lines on the canvas when object moves
  if curr_point and prev_point:
    cv2.line(canvas, prev_point, curr_point, (0, 0, 255), 5)  # Red ink line

  prev_point = curr_point

  # Combine live webcam feed and drawing canvas together
  combined = cv2.add(frame, canvas)

  # Display UI instructions
  cv2.putText(
      combined,
      "Press 'c' to Clear | Press 'q' to Quit",
      (10, 30),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.7,
      (255, 255, 255),
      2,
  )

  cv2.imshow('BCA AI Demo - Virtual Air Canvas', combined)

  key = cv2.waitKey(1) & 0xFF
  if key == ord('c'):
    canvas = np.zeros_like(frame)  # Clear canvas
  elif key == ord('q'):
    break

# Clean up webcam resources cleanly
cap.release()
cv2.waitKey(1)
cv2.destroyAllWindows()
cv2.waitKey(1)

Air Canvas Started! Wave a BLUE marker in front of the camera to draw.
Press 'c' to clear canvas. Press 'q' to quit.


-1